In [ ]:
  
import os
from tqdm import tqdm
import pandas as pd
import torch
import torch.nn as nn
from scipy.signal import resample
import scipy.io     
import numpy as np
import os
import gc
import mne
from sklearn.preprocessing import MinMaxScaler
from torch.utils.data import TensorDataset, TensorDataset, DataLoader
import torch.optim as optim
import copy 
from scipy.stats import pearsonr
import sys 
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
import pickle
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import pearsonr
import sys # For command-line arguments
print("All modules installed and imported successfully!")
from sklearn.metrics import mean_absolute_error, mean_squared_error





All modules installed and imported successfully!


In [ ]:
channels = [
    'Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'FC5', 'FC1', 'FC2', 
    'FC6', 'T7', 'C3', 'Cz', 'C4', 'T8', 'CP5', 'CP1', 'CP2', 'CP6', 
    'P7', 'P3', 'Pz', 'P4', 'P8', 'O1', 'Oz', 'O2', 'AF7', 'AF3', 
    'AF4', 'AF8', 'F5', 'F1', 'F2', 'F6', 'AFz', 'FT7', 'FC3', 'FC4', 
    'FT8', 'FCz', 'C5', 'C1', 'C2', 'C6', 'TP7', 'CP3', 'CPz', 'CP4', 
    'TP8', 'P5', 'P1', 'P2', 'P6', 'PO7', 'PO3', 'POz', 'PO4', 'PO6'
]
print(len(channels))

60


In [ ]:
import os
import numpy as np
import scipy.io
import pickle

# Set dataset folder
DATASET_FOLDER = "./data/"
leg_parts = [
'RightUpperLeg']

# Initialize list for kinematic data
kin_data_list = []
eeg_data_list=[]
eog_data_list=[]

kin_data_by_subject = []
eeg_data_by_subject = []
eog_data_by_subject = []

# Get all filenames in the dataset folder
all_files = set(os.listdir(DATASET_FOLDER))

def upsample(data,original_rate,target_rate):
    """Upsamples data using FFT-based resampling."""
    original_num_samples = data.shape[0]
    new_num_samples = int(original_num_samples * (target_rate / original_rate))
    resampled_data = scipy.signal.resample(data, new_num_samples, axis=0)
    return resampled_data



subject_id=1
for filename in sorted(all_files):
    if filename.endswith(".mat"):
        parts = filename.split('-')  # Correct delimiter

        if len(parts) != 3:
            print(f"Skipping {filename}: unexpected file format")
            continue
        # prev_subject=subject

        subject = parts[0]  # e.g., "AB_UH_01"
        trial = parts[1]    # e.g., "T00"
        data_type = parts[2].split('.')[0]  # e.g., "eeg" or "kin"

        if data_type != "kin":
            continue
        prev_subject_id=subject_id

        # Check for corresponding EEG file
        expected_eeg_filename = f"{subject}-{trial}-eeg.mat"
        if expected_eeg_filename not in all_files:
            print(f"Skipping {filename}: no matching EEG file ({expected_eeg_filename}) found.")
            continue

        # Load the .mat file
        file_path = os.path.join(DATASET_FOLDER, filename)
        eeg_file_path=os.path.join(DATASET_FOLDER, expected_eeg_filename)

        try:
            # Try to load the EEG file
            eeg_mat_data=scipy.io.loadmat(eeg_file_path, struct_as_record=False, squeeze_me=True)
            mat_data = scipy.io.loadmat(file_path, struct_as_record=False, squeeze_me=True)

            
        except PermissionError:
            # If a PermissionError occurs, print a warning and skip to the next loop iteration
            print(f"--> WARNING: Permission denied for file: {expected_eeg_filename}. Skipping this trial.")
            continue
        
        except Exception as e:
            print(f"--> WARNING: Could not load file {expected_eeg_filename} due to error: {e}. Skipping this trial.")
            continue
        

        # print(f"Attempting to load KIN file: {file_path}") 


        # Check sampling rate (kin.srate)
        if hasattr(mat_data["kin"], "srate"):
            srate = mat_data["kin"].srate
        #     print(f"Found kinematic file: {filename} with {srate} Hz sampling rate")

            current_trial_kin_data = []
            
            # Access the position struct
            position_struct = mat_data["kin"].data.position

            for part_name in leg_parts:
                part_data = getattr(position_struct, part_name)
                current_trial_kin_data.append(part_data)

            # Combine all parts into a single NumPy array for this trial
            # We concatenate along axis=1 to join the coordinate columns side-by-side
            combined_kin_data = np.concatenate(current_trial_kin_data, axis=1)


            # kin_data_list.append(downsample_to_30hz(combined_kin_data))
            print (f" before {combined_kin_data.shape}")
            # kin_data_list.append(upsample(combined_kin_data,srate,1000))
                
        

            subject_id=int(subject.split('_')[2])-1
            if(subject_id>prev_subject_id):
                np.save(f"RightUpperLeg/leftfoot_kin_data_for_subject{prev_subject_id}",kin_data_by_subject)
                kin_data_by_subject=[]
            kin_data_by_subject.append(upsample(combined_kin_data,srate,1000))
        
        eeg_obj = eeg_mat_data.get("eeg", None)
        if eeg_obj is not None:
            # Rawdata
            if hasattr(eeg_obj, "rawdata"):
                raw = eeg_obj.rawdata
                if isinstance(raw, np.ndarray):
                    subject_id=int(subject.split('_')[2])-1

                    if(subject_id>prev_subject_id):
                        output_filename=f"RightUpperLeg/eeg_data_for_subject{prev_subject_id}"
                        with open(output_filename, 'wb') as f:  # 'wb' means "write binary"
                            pickle.dump(eeg_data_by_subject, f)
                        eeg_data_by_subject=[]
                    eeg_data_by_subject.append(raw)
                    print(f"Appended rawdata from {expected_eeg_filename}, shape = {raw.shape}")


                else:
                    print(f"{expected_eeg_filename}: rawdata is not a NumPy array")

            # EOG data
            if hasattr(eeg_obj, "eogdata"):
                eog = eeg_obj.eogdata
                if isinstance(eog, np.ndarray):
                    if(subject_id>prev_subject_id):
                        output_filename=f"RightUpperLeg/eog_data_for_subject{prev_subject_id}"
                        with open(output_filename, 'wb') as f:  # 'wb' means "write binary"
                            pickle.dump(eog_data_by_subject, f)
                        eog_data_by_subject=[]
                    eog_data_by_subject.append(eog)
                    print(f"Appended rawdata from {expected_eeg_filename}, shape = {eog.shape}")



            
np.save(f"RightUpperLeg/leftfoot_kin_data_for_subject{prev_subject_id}",kin_data_by_subject)

output_filename=f"RightUpperLeg/eeg_data_for_subject{prev_subject_id}"
with open(output_filename, 'wb') as f:  # 'wb' means "write binary"
    pickle.dump(eeg_data_by_subject, f)
print("saved successfully")


output_filename=f"RightUpperLeg/eog_data_for_subject{prev_subject_id}"
with open(output_filename, 'wb') as f:  # 'wb' means "write binary"
    pickle.dump(eog_data_by_subject, f)


 before (6096, 3)
 before (6233, 3)
 before (5633, 3)
 before (5677, 3)
 before (5591, 3)
 before (5591, 3)
 before (5072, 3)
 before (5259, 3)
 before (5111, 3)
 before (5279, 3)
 before (4818, 3)
Skipping AB_UH_02-impedance.mat: unexpected file format
 before (4958, 3)


c:\Users\bigdipper\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\lib\npyio.py:501: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  arr = np.asanyarray(arr)


 before (4597, 3)
 before (4427, 3)
 before (4535, 3)
 before (4515, 3)
 before (4683, 3)
 before (4568, 3)
 before (4604, 3)
 before (4565, 3)
Skipping AB_UH_03-T10-kin.mat: no matching EEG file (AB_UH_03-T10-eeg.mat) found.
Skipping AB_UH_03-impedance.mat: unexpected file format
 before (5584, 3)
 before (5497, 3)
 before (5417, 3)
 before (5541, 3)
 before (5333, 3)
 before (5417, 3)
 before (5255, 3)
 before (5214, 3)
 before (5024, 3)
 before (5136, 3)
Skipping AB_UH_04-impedance.mat: unexpected file format
 before (5274, 3)
 before (4733, 3)
 before (4775, 3)
 before (5633, 3)
 before (5262, 3)
 before (5092, 3)
 before (4878, 3)
 before (2319, 3)
 before (2820, 3)
Skipping AB_UH_05-impedance.mat: unexpected file format
 before (5403, 3)
 before (5233, 3)
 before (5680, 3)
 before (5513, 3)
 before (5358, 3)
 before (5493, 3)
 before (5275, 3)
 before (5457, 3)
 before (5388, 3)
 before (5369, 3)
Skipping AB_UH_06-impedance.mat: unexpected file format
 before (10820, 3)
 before (